In [5]:
pip install scikit-learn pandas numpy matplotlib seaborn jupyterlab

Defaulting to user installation because normal site-packages is not writeable
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
  Using cached async_lru-2.0.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyter_lsp-2.3.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.17.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached setuptools-80.9.0-py3-none-any.wh


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\rohit\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import joblib

# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv("cardio.csv")

features = [
    "age_years",
    "bmi",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
]

X = df[features]

# ----------------------------
# Scale data
# ----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ----------------------------
# Train Isolation Forest
# ----------------------------
model = IsolationForest(
    n_estimators=300,
    contamination=0.05,   # top 5% anomalies
    random_state=42
)

model.fit(X_scaled)

# ----------------------------
# Save assets
# ----------------------------
joblib.dump(model, "cardio_anomaly_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("✅ Cardio anomaly model trained and saved.")


✅ Cardio anomaly model trained and saved.


In [9]:
import pandas as pd
import joblib

# Load model
model = joblib.load("cardio_anomaly_model.pkl")
scaler = joblib.load("scaler.pkl")

# Sample patient
patient = {
    "age_years": 55,
    "bmi": 33.8,
    "ap_hi": 170,
    "ap_lo": 105,
    "cholesterol": 3,
    "gluc": 2,
    "smoke": 1,
    "alco": 0,
    "active": 0
}

df = pd.DataFrame([patient])

# Scale + predict
X_scaled = scaler.transform(df)
flag = model.predict(X_scaled)[0]
score = model.decision_function(X_scaled)[0]

# ----------------------------
# Explanation layer
# ----------------------------
def generate_report(p, flag):
    if flag == -1:
        if p["ap_hi"] >= 160 or p["ap_lo"] >= 100:
            return "Blood pressure is in top 5% anomaly range → high cardiovascular risk detected"
        if p["bmi"] >= 30:
            return "BMI significantly above population norm → obesity-related risk detected"
        if p["cholesterol"] == 3:
            return "Cholesterol levels unusually high → lipid risk detected"
        return "Multiple vitals deviate from population baseline → clinical review recommended"
    else:
        return "Vitals fall within expected population distribution"

print("Anomaly Score:", round(score, 4))
print("Status:", "Anomalous" if flag == -1 else "Normal")
print("Patient Report:", generate_report(patient, flag))


Anomaly Score: -0.0975
Status: Anomalous
Patient Report: Blood pressure is in top 5% anomaly range → high cardiovascular risk detected
